In [60]:
import numpy as np
import spacy
from collections import Counter

In [56]:
texts = ["I like the red apple that grows on the apple tree.", 
        "Apple grows in the markets like a tree"]
D = len(texts)
print(f"{D=}")

D=2


### 1.Лемматизация

In [46]:
nlp = spacy.load("en_core_web_sm", exclude=['tok2vec', 'morphologizer', 'parser', 'attribute_ruler', 'ner'])
print(nlp.pipe_names)

def lemmatize(doc):
    words = []
    for token in doc:
        if (token.is_stop != True) and (token.is_punct != True) and\
            (token.is_space != True) and (token.is_digit != True):
            words.append(token.lemma_)
    return words

docs = list(nlp.pipe(texts))
lem_docs = [lemmatize(d) for d in docs]
print(f"{lem_docs=}")

['tagger', 'lemmatizer']
lem_docs=[['like', 'red', 'apple', 'grows', 'apple', 'tree'], ['apple', 'grows', 'markets', 'like', 'tree']]


### 2.Словарь (прямой и обратный)

In [55]:
vocab = {token:i for i, token in enumerate(sorted(set().union(*lem_docs)))}
inv_vocab = {i:token for token, i in vocab.items()}
W = len(vocab)
print(f"{vocab=}")
print(f"{inv_vocab=}")
print(f"{W=}")

vocab={'apple': 0, 'grows': 1, 'like': 2, 'markets': 3, 'red': 4, 'tree': 5}
inv_vocab={0: 'apple', 1: 'grows', 2: 'like', 3: 'markets', 4: 'red', 5: 'tree'}
W=6


### 3.Токенизированные документы

In [61]:
token_docs = [[vocab[t] for t in d ] for d in lem_docs ]
print(f"{token_docs=}")
[Counter(td) for td in token_docs]

token_docs=[[2, 4, 0, 1, 0, 5], [0, 1, 3, 2, 5]]


[Counter({0: 2, 2: 1, 4: 1, 1: 1, 5: 1}),
 Counter({0: 1, 1: 1, 3: 1, 2: 1, 5: 1})]

### 4.Подсчет токенов по классической схеме BOW до документам

In [67]:
def tokens_to_bow(tokens, W):
    counter = Counter(tokens)
    return [counter.get(token, 0) for token in range(W)]

n_dw = [tokens_to_bow(td, W) for td in token_docs]
print(f"{n_dw=}")

n_dw=[[2, 1, 1, 0, 1, 1], [1, 1, 1, 1, 0, 1]]


### 5.Подсчет токенов по схеме скользящего окна

In [66]:
sliding_window_range = (-1, 1) #  - левый и правый контекст по одному токену

def get_token_with_context(token_doc, swr):
    return [(i, token_doc[i + swr[0]: i + swr[1] + 1]) for i, _ in enumerate(token_doc) if i >= -swr[0] and i < len(token_doc) - swr[1]]

token_contexts = [get_token_with_context(td, sliding_window_range) for td in token_docs]
print(f"{token_contexts=}")

token_contexts=[[(1, [2, 4, 0]), (2, [4, 0, 1]), (3, [0, 1, 0]), (4, [1, 0, 5])], [(1, [0, 1, 3]), (2, [1, 3, 2]), (3, [3, 2, 5])]]


In [88]:
gamma = 0.3

#n_dw = np.empty((D, W), dtype=float)
rows = []

for num_d, d_tc in enumerate(token_contexts):
    print(f"Документ {num_d}")
    y = np.zeros(W, dtype=float)
    for tc in d_tc:
        print(f"{y=}")
        print(f"gamma * y = {gamma * y}")
        x = np.zeros(W, dtype=float)
        for token in tc[1]:
            x[token] += 1
        print(f"{x=}")
        print(f"(1 - gamma) * x = {(1 - gamma) * x}")
        y = gamma * y + (1 - gamma) * x
        print(f"{y=}")
        print()
    rows.append(y)
n_dw = np.vstack(rows)
print(f"{n_dw=}")

Документ 0
y=array([0., 0., 0., 0., 0., 0.])
gamma * y = [0. 0. 0. 0. 0. 0.]
x=array([1., 0., 1., 0., 1., 0.])
(1 - gamma) * x = [0.7 0.  0.7 0.  0.7 0. ]
y=array([0.7, 0. , 0.7, 0. , 0.7, 0. ])

y=array([0.7, 0. , 0.7, 0. , 0.7, 0. ])
gamma * y = [0.21 0.   0.21 0.   0.21 0.  ]
x=array([1., 1., 0., 0., 1., 0.])
(1 - gamma) * x = [0.7 0.7 0.  0.  0.7 0. ]
y=array([0.91, 0.7 , 0.21, 0.  , 0.91, 0.  ])

y=array([0.91, 0.7 , 0.21, 0.  , 0.91, 0.  ])
gamma * y = [0.273 0.21  0.063 0.    0.273 0.   ]
x=array([2., 1., 0., 0., 0., 0.])
(1 - gamma) * x = [1.4 0.7 0.  0.  0.  0. ]
y=array([1.673, 0.91 , 0.063, 0.   , 0.273, 0.   ])

y=array([1.673, 0.91 , 0.063, 0.   , 0.273, 0.   ])
gamma * y = [0.5019 0.273  0.0189 0.     0.0819 0.    ]
x=array([1., 1., 0., 0., 0., 1.])
(1 - gamma) * x = [0.7 0.7 0.  0.  0.  0.7]
y=array([1.2019, 0.973 , 0.0189, 0.    , 0.0819, 0.7   ])

Документ 1
y=array([0., 0., 0., 0., 0., 0.])
gamma * y = [0. 0. 0. 0. 0. 0.]
x=array([1., 1., 0., 1., 0., 0.])
(1 - gamma) 